# Which layers hold redundant information?

This notebook reads the expanded activation caches written by
`scripts/run_activation_caching_expanded_selected_acts.sh` (`layer_out/17` … `layer_out/35`
at prompt positions -2 and -1) and asks, for one dataset folder, **which layers are
near-duplicates of each other** — so that downstream work can keep one layer per
redundant group instead of all nineteen.

Five matrices, each answering a different sense of "redundant":

| Matrix | Question it answers |
|---|---|
| **Linear CKA** | Do two layers induce the same geometry over prompts? |
| **Linear predictivity (R²)** | Can layer *j* be linearly reconstructed from layer *i*? (asymmetric) |
| **Subspace overlap** | Do the dominant PCA subspaces of two layers span the same directions? |
| **PLS-1 correlation** | Do two layers encode the *time-horizon* direction with the same sign and ordering? |
| **Residual CKA** | After the 6 PLS components are removed, is the *remaining* structure still shared? |

Alongside them: per-layer decodability of `log10_time_horizon_months`, effective
dimensionality, and a clustering that turns the CKA matrix into concrete layer groups.

A layer is a good candidate for dropping when it is high on CKA / predictivity /
subspace overlap with a neighbour **and** does not beat that neighbour on target R².


## 1. Setup

For a fresh Colab runtime, uncomment the clone and authentication lines.


In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env


In [ ]:
import gc
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from google.cloud import storage
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')


## 2. Configure

`ACTIVATIONS_GCS_URI` points at one `expanded_*` folder. `MAX_BATCH_FILES` and `MAX_ROWS`
bound the download and the working set: every matrix here is computed from an
`MAX_ROWS x hidden` sample per layer, and the CKA step holds one `MAX_ROWS x MAX_ROWS`
Gram matrix per layer (19 layers x 2000² x 4 bytes ≈ 300 MB), so raise `MAX_ROWS` only
with that in mind.


In [ ]:
ACTIVATIONS_GCS_URI = 'gs://temporal-research-bucket/expanded_selected_acts'
DATA_DIR = repo_root / 'data' / 'layer_redundancy'
PROJECT_ID = os.getenv('GCP_PROJECT_ID')

MAX_BATCH_FILES: int | None = 8      # None downloads the whole folder.
MAX_ROWS = 2_000                     # Prompts sampled for every matrix.
OVERWRITE = False
DOWNLOAD_WORKERS = 8
RANDOM_SEED = 0

PLS_COMPONENTS = 6                   # Matches scripts/fit_expanded_pls_residual_pca.py.
PCA_COMPONENTS = 64                  # Reduced space for linear predictivity.
SUBSPACE_COMPONENTS = 16             # Top-k subspace compared across layers.
REDUNDANCY_CKA_THRESHOLD = 0.95      # CKA above this counts as redundant when clustering.

rng = np.random.default_rng(RANDOM_SEED)


### Palette

One hue, light → dark, for every magnitude matrix (CKA, predictivity, overlap); a
blue↔red diverging pair with a neutral gray midpoint for the signed correlation matrix;
the fixed categorical order for line series. No rainbow scales — a rainbow makes equal
steps in value look like unequal steps in color.


In [ ]:
SEQUENTIAL_BLUE = [
    [0.00, '#f4f8fe'], [0.15, '#cde2fb'], [0.30, '#9ec5f4'], [0.45, '#6da7ec'],
    [0.60, '#3987e5'], [0.75, '#256abf'], [0.90, '#184f95'], [1.00, '#0d366b'],
]
DIVERGING_BLUE_RED = [
    [0.00, '#0d366b'], [0.20, '#256abf'], [0.40, '#9ec5f4'], [0.50, '#f0efec'],
    [0.60, '#f6b0af'], [0.80, '#e34948'], [1.00, '#8f1f1e'],
]
SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
TEXT_PRIMARY, TEXT_SECONDARY, GRID = '#0b0b0b', '#52514e', '#e6e5e1'


def style_figure(fig, title, subtitle=None, height=480):
    """Apply the shared chart frame: recessive axes, quiet grid, text-token labels."""
    heading = title if subtitle is None else f'{title}<br><sup>{subtitle}</sup>'
    fig.update_layout(
        title={'text': heading, 'font': {'size': 17, 'color': TEXT_PRIMARY}, 'x': 0, 'xanchor': 'left'},
        template='simple_white',
        height=height,
        margin={'l': 70, 'r': 30, 't': 80, 'b': 60},
        font={'color': TEXT_SECONDARY, 'size': 12},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'left', 'x': 0},
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID, ticks='outside', tickcolor=GRID)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, ticks='outside', tickcolor=GRID)
    return fig


def matrix_figure(matrix, labels, title, subtitle, colorbar_title, *, diverging=False, zmin=None, zmax=None):
    """Render one layer-by-layer matrix as a heatmap with per-cell hover."""
    colorscale = DIVERGING_BLUE_RED if diverging else SEQUENTIAL_BLUE
    limit = float(np.nanmax(np.abs(matrix))) if diverging else None
    fig = go.Figure(
        go.Heatmap(
            z=matrix,
            x=labels,
            y=labels,
            colorscale=colorscale,
            zmid=0 if diverging else None,
            zmin=-limit if diverging else zmin,
            zmax=limit if diverging else zmax,
            colorbar={'title': {'text': colorbar_title, 'side': 'right'}, 'thickness': 14, 'outlinewidth': 0},
            hovertemplate='row %{y}<br>column %{x}<br>value %{z:.3f}<extra></extra>',
        )
    )
    style_figure(fig, title, subtitle, height=620)
    fig.update_yaxes(autorange='reversed', showgrid=False)
    fig.update_xaxes(side='bottom')
    fig.update_layout(width=700)
    return fig


def line_figure(x_values, series, title, subtitle, y_title, x_title='Layer'):
    """Render one or more per-layer curves; a single series carries no legend box."""
    fig = go.Figure()
    for index, (name, values) in enumerate(series.items()):
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=values,
                name=name,
                mode='lines+markers',
                line={'width': 2, 'color': SERIES_COLORS[index % len(SERIES_COLORS)]},
                marker={'size': 8},
                hovertemplate=f'{name}<br>layer %{{x}}<br>%{{y:.3f}}<extra></extra>',
            )
        )
    style_figure(fig, title, subtitle)
    fig.update_layout(showlegend=len(series) > 1)
    fig.update_xaxes(title_text=x_title, dtick=1)
    fig.update_yaxes(title_text=y_title)
    return fig


## 3. Download a sample of batches

Only the first `MAX_BATCH_FILES` batch files are fetched — enough rows for every matrix
here, and far cheaper than a full expanded folder (which runs to tens of GB).


In [ ]:
def parse_gcs_uri(uri):
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got {uri!r}')
    bucket, separator, prefix = uri[5:].partition('/')
    if not bucket or not separator or not prefix.strip('/'):
        raise ValueError(f'GCS URI must contain a bucket and prefix: {uri!r}')
    return bucket, prefix.strip('/')


client = storage.Client(project=PROJECT_ID)
bucket_name, activation_prefix = parse_gcs_uri(ACTIVATIONS_GCS_URI)
blobs = sorted(
    (
        blob for blob in client.bucket(bucket_name).list_blobs(prefix=activation_prefix + '/')
        if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
    ),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(f'No activation batches found below {ACTIVATIONS_GCS_URI}')
if MAX_BATCH_FILES is not None:
    blobs = blobs[:MAX_BATCH_FILES]

batch_dir = DATA_DIR / activation_prefix
batch_dir.mkdir(parents=True, exist_ok=True)


def download_batch(blob):
    destination = batch_dir / Path(blob.name).name
    if OVERWRITE or not destination.exists():
        blob.download_to_filename(str(destination))
    return destination


with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    batch_paths = sorted(tqdm(
        executor.map(download_batch, blobs),
        total=len(blobs),
        desc='Downloading batches',
    ))

print(f'Source: {ACTIVATIONS_GCS_URI}')
print(f'Local batches: {len(batch_paths):,} in {batch_dir}')


## 4. Load activations per layer and position

Each batch holds every cached layer at both positions. Rows are kept only when the prompt
declares a time horizon, since `log10_time_horizon_months` is the target used below.
Activations are mean-centered per layer, which is what both CKA and PCA assume.


In [ ]:
UNIT_TO_MONTHS = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1.0, 'year': 12.0, 'decade': 120.0, 'century': 1200.0, 'millennium': 12000.0,
}
UNIT_TO_MONTHS.update({f'{unit}s': value for unit, value in list(UNIT_TO_MONTHS.items())})
UNIT_TO_MONTHS['centuries'] = 1200.0
UNIT_TO_MONTHS['millennia'] = 12000.0


def horizon_months(metadata):
    value = metadata.get('base_value', metadata.get('value'))
    unit = metadata.get('base_unit', metadata.get('unit'))
    if value in (None, 'N/A') or unit in (None, 'N/A'):
        return None
    return float(value) * UNIT_TO_MONTHS[str(unit).lower()]


layer_rows = {}
target_parts = []
metadata_rows = []
positions = None
layers = None

for path in tqdm(batch_paths, desc='Loading batches'):
    payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
    batch_positions = list(payload['positions'])
    batch_layers = sorted(
        int(key.split('/')[-1]) for key in payload['activations']
    )
    if positions is None:
        positions, layers = batch_positions, batch_layers
    elif batch_positions != positions or batch_layers != layers:
        raise ValueError(f'{path} disagrees with earlier batches on layers or positions.')

    keep = [
        offset for offset, metadata in enumerate(payload['prompt_metadata'])
        if horizon_months(metadata) is not None
    ]
    if not keep:
        continue
    rows = torch.as_tensor(keep, dtype=torch.long)
    for layer in layers:
        tensor = payload['activations'][f'layer_out/{layer}'].index_select(0, rows).to(torch.float32)
        for position_index in range(len(positions)):
            layer_rows.setdefault((layer, position_index), []).append(tensor[:, position_index, :].numpy())
    target_parts.append(np.array([
        np.log10(horizon_months(payload['prompt_metadata'][offset])) for offset in keep
    ]))
    metadata_rows.extend(payload['prompt_metadata'][offset] for offset in keep)
    del payload
    gc.collect()

target_all = np.concatenate(target_parts)
row_count = len(target_all)
sample_rows = np.sort(rng.choice(row_count, size=min(MAX_ROWS, row_count), replace=False))
target = target_all[sample_rows]

activations = {}
for key, parts in layer_rows.items():
    stacked = np.concatenate(parts, axis=0)[sample_rows]
    activations[key] = stacked - stacked.mean(axis=0, keepdims=True)
del layer_rows, target_parts
gc.collect()

position_names = {index: f'position {value}' for index, value in enumerate(positions)}
layer_labels = [str(layer) for layer in layers]
hidden_size = activations[(layers[0], 0)].shape[1]
print(f'Layers: {layers[0]}..{layers[-1]} ({len(layers)})   positions: {positions}')
print(f'Rows kept: {row_count:,}; sampled for analysis: {len(sample_rows):,}; hidden size: {hidden_size:,}')
print(f'Target log10_time_horizon_months range: {target.min():.2f} .. {target.max():.2f}')


## 5. Redundancy matrices

**Linear CKA** compares the prompt-by-prompt similarity structure two layers induce. It is
invariant to rotation and isotropic scaling, so it answers "same geometry?" rather than
"same coordinates?". 1.0 means the two layers are the same representation up to those
transformations — the strongest form of redundancy.


In [ ]:
def centered_gram(features):
    """Gram matrix of already-centered features; equals the double-centered kernel."""
    return features @ features.T


def cka_matrix(features_by_layer, layer_order):
    """Linear CKA between every pair of layers, computed from cached Gram matrices."""
    grams = [centered_gram(features_by_layer[layer]) for layer in layer_order]
    norms = np.array([np.sqrt((gram * gram).sum()) for gram in grams])
    size = len(layer_order)
    matrix = np.ones((size, size))
    for i in range(size):
        for j in range(i + 1, size):
            value = float((grams[i] * grams[j]).sum() / (norms[i] * norms[j]))
            matrix[i, j] = matrix[j, i] = value
    del grams
    gc.collect()
    return matrix


cka_by_position = {}
for position_index in range(len(positions)):
    features = {layer: activations[(layer, position_index)] for layer in layers}
    cka_by_position[position_index] = cka_matrix(features, layers)
    print(f'{position_names[position_index]}: CKA matrix computed.')


In [ ]:
for position_index, matrix in cka_by_position.items():
    matrix_figure(
        matrix,
        layer_labels,
        f'Linear CKA between layers — {position_names[position_index]}',
        'Bright blocks along the diagonal are runs of near-duplicate layers',
        'CKA',
        zmin=float(min(m.min() for m in cka_by_position.values())),
        zmax=1.0,
    ).show()


### Where does the representation actually change?

The same numbers read as a curve: CKA between each layer and its predecessor. Flat stretches
near 1.0 are exactly the redundant runs — the residual stream is being carried forward with
little change. Dips mark layers that do real work.


In [ ]:
adjacent_series = {
    position_names[position_index]: [np.nan] + [
        float(matrix[index - 1, index]) for index in range(1, len(layers))
    ]
    for position_index, matrix in cka_by_position.items()
}
line_figure(
    layers,
    adjacent_series,
    'CKA with the previous layer',
    'Values near 1.0 mean the layer adds little the previous layer did not already hold',
    'CKA(layer, layer - 1)',
).show()


### Do the two cached positions duplicate each other?

Redundancy across *positions* matters as much as across layers: if position -2 and -1 carry
the same geometry at a given layer, concatenating both doubles the feature width for little
gain.


In [ ]:
def cross_gram_cka(features_a, features_b):
    gram_a, gram_b = centered_gram(features_a), centered_gram(features_b)
    numerator = float((gram_a * gram_b).sum())
    denominator = float(np.sqrt((gram_a * gram_a).sum() * (gram_b * gram_b).sum()))
    return numerator / denominator


position_cka = [
    cross_gram_cka(activations[(layer, 0)], activations[(layer, 1)]) for layer in tqdm(layers, desc='Position CKA')
]
line_figure(
    layers,
    {f'CKA({position_names[0]}, {position_names[1]})': position_cka},
    'Agreement between the two cached prompt positions',
    'High values mean the second position adds little beyond the final token',
    'CKA between positions',
).show()


### Linear predictivity — can one layer be *reconstructed* from another?

CKA is symmetric; reconstruction is not. Each layer is reduced to its top `PCA_COMPONENTS`
components, then row *i* / column *j* holds the held-out R² of predicting layer *j*'s
reduced code from layer *i*'s by ordinary least squares. A high row means "everything later
is already linearly present here"; a high column means "this layer is fully explained by
others".


In [ ]:
position_index = len(positions) - 1  # Final prompt token.
component_limit = min(PCA_COMPONENTS, len(sample_rows) - 1, hidden_size)
subspace_limit = min(SUBSPACE_COMPONENTS, component_limit)
reduced = {}
explained = {}
for layer in tqdm(layers, desc='Per-layer PCA'):
    pca = PCA(n_components=component_limit, svd_solver='randomized', random_state=RANDOM_SEED)
    reduced[layer] = pca.fit_transform(activations[(layer, position_index)])
    explained[layer] = pca.explained_variance_

split = len(sample_rows) // 2
train_rows = np.arange(split)
test_rows = np.arange(split, len(sample_rows))


def linear_r2(source, destination):
    design = np.column_stack([source[train_rows], np.ones(len(train_rows))])
    coefficients, *_ = np.linalg.lstsq(design, destination[train_rows], rcond=None)
    predicted = np.column_stack([source[test_rows], np.ones(len(test_rows))]) @ coefficients
    actual = destination[test_rows]
    residual_ss = float(np.square(actual - predicted).sum())
    total_ss = float(np.square(actual - actual.mean(axis=0)).sum())
    return 1.0 - residual_ss / total_ss


predictivity = np.zeros((len(layers), len(layers)))
for i, source_layer in enumerate(tqdm(layers, desc='Linear predictivity')):
    for j, destination_layer in enumerate(layers):
        predictivity[i, j] = 1.0 if i == j else linear_r2(reduced[source_layer], reduced[destination_layer])

matrix_figure(
    predictivity,
    layer_labels,
    f'Linear predictivity R² — {position_names[position_index]}',
    f'Row = predictor layer, column = predicted layer; top {component_limit} PCA components, held-out split',
    'R²',
    zmin=0.0,
    zmax=1.0,
).show()


### Subspace overlap — do the dominant directions coincide?

Mean squared cosine of the principal angles between each pair of leading PCA subspaces
(`SUBSPACE_COMPONENTS`, capped by the sample size and hidden width). 1.0 means the two layers' leading subspaces span the same directions; this is
stricter than CKA about *which* directions matter and ignores how variance is distributed
inside the subspace.


In [ ]:
bases = {}
for layer in tqdm(layers, desc='Subspace bases'):
    pca = PCA(n_components=subspace_limit, svd_solver='randomized', random_state=RANDOM_SEED)
    pca.fit(activations[(layer, position_index)])
    bases[layer] = pca.components_.T  # hidden x k, orthonormal columns.

overlap = np.eye(len(layers))
for i in range(len(layers)):
    for j in range(i + 1, len(layers)):
        singular = np.linalg.svd(bases[layers[i]].T @ bases[layers[j]], compute_uv=False)
        overlap[i, j] = overlap[j, i] = float(np.square(singular).mean())

matrix_figure(
    overlap,
    layer_labels,
    f'Principal-subspace overlap (top {subspace_limit} components)',
    'Mean squared cosine of principal angles; 1.0 = identical leading subspace',
    'overlap',
    zmin=0.0,
    zmax=1.0,
).show()


## 6. Redundancy *about the target*

Two layers can be geometrically similar yet differ in how well the time horizon can be read
out of them — and that is what decides which one to keep. Below: per-layer held-out R² of a
6-component PLS regression on `log10_time_horizon_months`, for each position separately and
for the concatenated pair used by `scripts/fit_expanded_pls_residual_pca.py`.


In [ ]:
def pls_fit(features, components=PLS_COMPONENTS):
    model = PLSRegression(n_components=components, scale=False)
    model.fit(features[train_rows], target[train_rows])
    return model


target_r2 = {}
pls_models = {}
for position_key in range(len(positions)):
    scores = []
    for layer in tqdm(layers, desc=f'PLS {position_names[position_key]}'):
        model = pls_fit(activations[(layer, position_key)])
        pls_models[(layer, position_key)] = model
        scores.append(float(model.score(activations[(layer, position_key)][test_rows], target[test_rows])))
    target_r2[position_names[position_key]] = scores

concatenated_r2 = []
for layer in tqdm(layers, desc='PLS both positions'):
    features = np.concatenate(
        [activations[(layer, index)] for index in range(len(positions))], axis=1
    )
    model = pls_fit(features)
    concatenated_r2.append(float(model.score(features[test_rows], target[test_rows])))
target_r2['both positions concatenated'] = concatenated_r2

line_figure(
    layers,
    target_r2,
    f'Held-out R² of a {PLS_COMPONENTS}-component PLS on log10_time_horizon_months',
    'Layers on a redundant plateau can be dropped in favour of the best-scoring member',
    'R² (held-out half)',
).show()


### Do layers order the horizon the same way?

Correlation of the first PLS score across layers. Sign matters — a PLS direction is only
defined up to sign, so blue cells simply mean the two layers found the same axis pointing
opposite ways. What is informative is the **magnitude**: near ±1 means the layers rank prompts
identically along the horizon axis, i.e. redundant *for this task* even if their full
geometries differ.


In [ ]:
first_scores = np.column_stack([
    pls_models[(layer, position_index)].transform(activations[(layer, position_index)])[:, 0]
    for layer in layers
])
score_correlation = np.corrcoef(first_scores, rowvar=False)

matrix_figure(
    score_correlation,
    layer_labels,
    'Correlation of the first PLS score across layers',
    'Sign is arbitrary per layer; |r| near 1 means the same prompt ordering along the horizon axis',
    'r',
    diverging=True,
).show()


### Residual CKA — shared structure that is *not* the horizon

The 6 PLS components are deflated out of every layer, then CKA is recomputed. Comparing this
with the first heatmap separates two very different situations: layers that agree only
because they both encode the horizon (block fades here) versus layers that are wholesale
copies of each other (block survives).


In [ ]:
residuals = {}
for layer in tqdm(layers, desc='PLS residuals'):
    features = activations[(layer, position_index)]
    model = pls_models[(layer, position_index)]
    residuals[layer] = features - model.inverse_transform(model.transform(features))

residual_cka = cka_matrix(residuals, layers)
matrix_figure(
    residual_cka,
    layer_labels,
    'Linear CKA after removing the 6 PLS components',
    'Blocks that survive are duplication beyond the time-horizon signal',
    'CKA (residual)',
    zmin=float(residual_cka.min()),
    zmax=1.0,
).show()

difference = cka_by_position[position_index] - residual_cka
matrix_figure(
    difference,
    layer_labels,
    'CKA lost by removing the PLS components',
    'Positive cells: the layers agreed mainly because both carry the horizon signal',
    'ΔCKA',
    diverging=True,
).show()


### Effective dimensionality

Participation ratio of each layer's PCA spectrum, `(Σλ)² / Σλ²` — roughly "how many directions
carry the variance". A layer whose effective dimensionality collapses toward its neighbours'
while CKA stays high is carrying no new degrees of freedom.


In [ ]:
participation_ratio = [
    float(np.square(explained[layer].sum()) / np.square(explained[layer]).sum()) for layer in layers
]
line_figure(
    layers,
    {'participation ratio': participation_ratio},
    f'Effective dimensionality of the top {component_limit} components',
    f'{position_names[position_index]}; higher means variance is spread over more directions',
    'participation ratio',
).show()


## 7. From matrices to a layer shortlist

The CKA matrix is turned into a distance (`1 - CKA`), clustered with average linkage, and cut
at `REDUNDANCY_CKA_THRESHOLD`. Every cluster is a set of mutually redundant layers; the
representative is the member with the highest held-out target R².


In [ ]:
cka_matrix_final = cka_by_position[position_index]
distance = np.clip(1.0 - cka_matrix_final, 0.0, None)
np.fill_diagonal(distance, 0.0)
linkage_matrix = linkage(squareform(distance, checks=False), method='average')
cluster_ids = fcluster(linkage_matrix, t=1.0 - REDUNDANCY_CKA_THRESHOLD, criterion='distance')

dendrogram = ff.create_dendrogram(
    np.zeros((len(layers), 1)),
    labels=layer_labels,
    linkagefun=lambda _: linkage_matrix,
    colorscale=[SERIES_COLORS[0]] * 8,
)
style_figure(
    dendrogram,
    'Layer clustering on 1 - CKA',
    f'Average linkage; merges below {1 - REDUNDANCY_CKA_THRESHOLD:.2f} are redundant at the chosen threshold',
    height=420,
)
dendrogram.update_yaxes(title_text='1 - CKA')
dendrogram.update_xaxes(title_text='Layer')
dendrogram.show()


In [ ]:
layer_r2 = dict(zip(layers, target_r2[position_names[position_index]]))
summary = pd.DataFrame({
    'layer': layers,
    'cluster': cluster_ids,
    'target_r2': [layer_r2[layer] for layer in layers],
    'cka_with_previous': adjacent_series[position_names[position_index]],
    'participation_ratio': participation_ratio,
    'mean_cka_with_others': [
        float((cka_matrix_final[index].sum() - 1.0) / (len(layers) - 1)) for index in range(len(layers))
    ],
})
summary['is_cluster_representative'] = summary['target_r2'] == summary.groupby('cluster')['target_r2'].transform('max')

representatives = summary.loc[summary['is_cluster_representative'], 'layer'].tolist()
print(f'{summary["cluster"].nunique()} cluster(s) at CKA >= {REDUNDANCY_CKA_THRESHOLD}')
print(f'Suggested layers to keep: {representatives}')
print(f'Redundant layers to drop: {[layer for layer in layers if layer not in representatives]}')
summary.round(4)


### Optional: write the shortlist out

Uncomment to save the per-layer summary next to the PLS results.


In [ ]:
# output_path = repo_root / 'results' / 'expanded_pls' / f'{activation_prefix}_layer_redundancy.csv'
# output_path.parent.mkdir(parents=True, exist_ok=True)
# summary.to_csv(output_path, index=False)
# print(f'Wrote {output_path}')


## 8. Reading these matrices

- **Bright square blocks on the CKA diagonal** are the headline result: contiguous layer runs
  that hold the same geometry. Keep one layer per block.
- **A high CKA block that survives in the residual CKA** is genuine duplication. A block that
  disappears means the layers only agreed through the horizon signal — they may still differ
  in what else they encode.
- **Asymmetry in the predictivity matrix** tells you the direction of the redundancy: if layer
  *i* predicts *j* well but not the reverse, *j* is closer to a compression of *i*.
- **High |r| in the PLS-score correlation with differing target R²** means both layers hold
  the horizon axis but one reads it out more cleanly — prefer the higher R².
- Re-run with a different `ACTIVATIONS_GCS_URI` to check whether the redundant blocks are a
  property of the model or of one dataset's phrasing. Blocks that persist across datasets are
  the safe ones to prune.
